# 126 — Handoffs y transferencia de contexto

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Handoff**: transferencia atómica de la *propiedad* de una tarea en curso, de A a B,
con un payload de contexto destilado. Lo inicia el agente al reconocer su límite —
a diferencia del router, que decide antes de empezar; y del subagente, donde el control
vuelve al padre.

**Payload con esquema**: `goal`, `reason`, `verified_facts` (separados de hipótesis),
`work_done`, `work_remaining`, `constraints`, `artifacts` por URI. Destilar ≠ copiar
historial: viaja lo accionable.

**Escalada**: handoff hacia mayor autoridad; añade `urgency` y `decision_requested`.
Regla de seguridad: contador de saltos + corte a humano si hay ciclos (A→B→A).


## 🧮 Ejemplo de referencia

Ticket "pago falla con 502" → triage verifica hechos → handoff a infra con payload de
~250 tokens (el historial completo eran ~3 000: viaja el 8 % del texto y el 100 % de lo
accionable) → infra detecta regresión del deploy → escalada a humano con
`decision_requested = "aprobar rollback"` → aprobado → devolución a triage y cierre.

En cada paso hay exactamente **un** dueño de la tarea, y cada traspaso queda
registrado con su `reason`.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("multiagent", seed=126)
show(result)


## Reflexión

1. En el payload de ejemplo, ¿qué pasaría si `verified_facts` y las hipótesis viajaran mezcladas en un solo campo `notes`? Describe un fallo concreto que B podría cometer.
2. El laboratorio consolida workers que nunca se traspasan la tarea. ¿En qué punto del flujo supervisor→workers tendría sentido un handoff worker→worker y qué campos del payload serían críticos?
3. ¿Qué métrica registrarías para detectar que los handoffs de tu sistema están destruyendo contexto (el clásico "teléfono roto")?
